In [1]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
devtools::load_all('utils/modules/R/prstools')
library(argparse)

i Loading PRStools

Loading required package: bigsnpr

Loading required package: bigstatsr

Loading required package: data.table

Loading required package: bigassertr



In [2]:
args <- list(
    chrom = "chr21",
    pred = "data/prs/hapmap/ukb_500k/ukb_hapmap_500k_eur_chr21.bed",
    ldsc = "data/prs/ldsc/ldsc_Direct_Bilirubin_residual.rds",
    ld_dir = "data/prs/hapmap/ld/matrix",
    method = 'auto',
    out_prefix = "data/prs/scores/auto/prs_auto_Direct_Bilirubin_residual",
    tmp_bfile = "data/prs/scores/auto/prs_auto_Direct_Bilirubin_residual_chr21.bfile"
)

In [3]:
# check args$chr
stopifnot(!is.null(args$chr))
stopifnot(file.exists(args$pred))
stopifnot(file.exists(args$ldsc))
stopifnot(!file.exists(args$tmp_bfile))
stopifnot(dir.exists(args$ld_dir))
stopifnot(dir.exists(dirname(args$out_prefix)))
stopifnot(args$method %in% c('inf', 'auto'))

# setup parallel environment
NCORES <- max(1, nb_cores())
bigparallelr::assert_cores(NCORES)
msg <- paste0("Running ldpred2 with ",NCORES," cores [",Sys.time(),"]")
write(msg,stdout())

# laod ldsc and qced GWAS
ldsc <- readRDS(args$ldsc)
gwas <- ldsc$gwas
h2 <- ldsc$h2_est

# Estimate h2 chromosome-wide
N_total <- nrow(gwas)
N_chr <- sum(gwas$chr == args$chr)
h2_init <- h2 * (N_chr / N_total)

# check estimates
stopifnot(h2_init > 0)
stopifnot(!is.null(gwas))

Running ldpred2 with 1 cores [2022-05-20 10:29:02]


In [4]:
h2_init

[1] 0.005923561

In [5]:
# get SNP correlations and LD
snp <- get_single_ld_matrix(gwas, chr = args$chrom, ld_dir = args$ld_dir)

# match GWAS with snp-correlation map
gwas <- gwas[na.omit(match(snp$map$marker, gwas$marker)), ]

In [6]:
# check that LD-matrix markers and gwas markers have overlap
# Check that ordering of markers are actually matching
stopifnot(all(gwas$marker %in% snp$map$marker))
stopifnot(all(snp$map$marker %in% gwas$marker))
stopifnot(sum(gwas$marker == snp$map$marker) / nrow(gwas) == 1)

# load data to be used for prediction
bfile <- tempfile(tmpdir = dirname(args$out_prefix))
pred <- load_bigsnp_from_bed(args$pred)
pred <- match_bigsnp_with_gwas(obj=pred, gwas=gwas, bfile=args$tmp_bfile)
genotypes <- pred$genotypes
indicies <- pred$gwas_indicies

stopifnot(!is.null(genotypes))
stopifnot(!is.null(indicies))

In [7]:
# count variants with missing GTs
cols <- genotypes$`.->ncol`
rows <- genotypes$`.->nrow`
sum_rows <- lapply(1:rows, function(i)
    return(sum(is.na(genotypes[i,])))
)

missing_gt <- sum(unlist(sum_rows))
pct <- paste0(round(missing_gt/rows, 4)*100)


In [14]:
pct <- round(missing_gt/rows, 4)*100

In [15]:
pct

[1] 2.75

In [18]:
write("Running multi_auto", stderr())
multi_auto <- snp_ldpred2_auto(
    corr = snp$corr,
    df_beta = gwas,
    h2_init = h2_init,
    vec_p_init = seq_log(1e-4, 0.9, 30),
    ncores = NCORES)

# save data chains
#multi_auto_path <- paste0(args$out_prefix,'_chains.rda')
#multi_auto <- readRDS(multi_auto_path)

In [19]:
str(multi_auto)

List of 30
 $ :List of 9
  ..$ beta_est   : num [1:16307] -4.76e-05 -6.45e-06 -6.51e-06 -5.77e-06 -1.52e-06 ...
  ..$ postp_est  : num [1:16307] 0.214 0.212 0.212 0.212 0.211 ...
  ..$ sample_beta: num[1:16307, 0 ] 
  ..$ p_est      : num 0.214
  ..$ h2_est     : num 0.00123
  ..$ path_p_est : num [1:1500] 7.25e-05 1.63e-04 3.75e-04 1.42e-04 2.95e-04 ...
  ..$ path_h2_est: num [1:1500] 0.0001 0.0001 0.00019 0.000104 0.000199 ...
  ..$ h2_init    : num 0.00592
  ..$ p_init     : num 1e-04
 $ :List of 9
  ..$ beta_est   : num [1:16307] -4.60e-05 -6.14e-06 -6.27e-06 -5.53e-06 -1.41e-06 ...
  ..$ postp_est  : num [1:16307] 0.152 0.149 0.149 0.149 0.148 ...
  ..$ sample_beta: num[1:16307, 0 ] 
  ..$ p_est      : num 0.151
  ..$ h2_est     : num 0.0012
  ..$ path_p_est : num [1:1500] 5.98e-05 1.49e-05 1.17e-05 4.94e-05 2.85e-04 ...
  ..$ path_h2_est: num [1:1500] 1e-04 1e-04 1e-04 1e-04 1e-04 ...
  ..$ h2_init    : num 0.00592
  ..$ p_init     : num 0.000137
 $ :List of 9
  ..$ beta_est   : 

In [20]:
test <- snp_fastImputeSimple(genotypes)

ERROR: Error: You don't have write permissions for this FBM.


In [ ]:
# get estimates with indicies corresponding to pred genotypes
beta_auto <- sapply(multi_auto, function(auto){
      auto$beta_est})

# perform matrix multiplication
pred_auto <- big_prodMat(
    test,
    beta_auto,
    ncores = NCORES)

In [53]:
# quality controls on chains
na_rows <- rowSums(is.na(pred_auto)) > 0 
na_rows_pct <- round(100*(sum(na_rows) / nrow(pred_auto)), 2)
write(paste0(na_rows_pct,"% of chain rows contains NAs (", args$out_prefix, ")" ), stdout())





0% of chain rows contains NAs (data/prs/scores/auto/prs_auto_BMI_chr21)


In [55]:
sum(M)

[1] 10488

In [ ]:
sc <- apply(pred_auto, 2, sd, na.rm = TRUE)
keep <- abs(sc - median(sc)) < 3 * mad(sc)
final_beta_auto <- rowMeans(beta_auto[, keep], na.rm = TRUE)

In [ ]:
# get final predicton
final_pred_auto <- big_prodVec(
   genotypes,
   final_beta_auto,
   ncores = NCORES)

final_pred <- final_pred_auto